In [1]:
import streamlit as st
from agno.agent import Agent
from agno.knowledge.pdf import PDFKnowledgeBase, PDFReader
from agno.vectordb.qdrant import Qdrant
from agno.tools.duckduckgo import DuckDuckGoTools
from agno.models.openai import OpenAIChat
from agno.embedder.openai import OpenAIEmbedder
import tempfile
import os
from agno.document.chunking.document import DocumentChunking


In [2]:
import getpass
import os
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langchain_deepseek import ChatDeepSeek

deepseek_api_key = "sk-f46c7e2053764299ace45b6e3d98de77"
base_url = "https://api.deepseek.com"

# model = ChatOpenAI(model="gpt-4o")
if not os.getenv("DEEPSEEK_API_KEY"):
    os.environ["DEEPSEEK_API_KEY"] = deepseek_api_key

llm = ChatDeepSeek(
    model="deepseek-chat",
    temperature=0,
    max_tokens=None,
    timeout=None,
    max_retries=2,
    # other params...
)


# @tool
# def magic_function(input: int) -> int:
#     """Applies a magic function to an input."""
#     return input + 2


# tools = [magic_function]


# query = "what is the value of magic_function(77)?"


In [3]:
import os
import tempfile
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings




def process_pdf_to_faiss(pdf_bytes, index_path="faiss_index"):
    """
    将 PDF 文件切块、生成向量，并存入本地 FAISS
    """
    # 1. 临时保存 PDF
    with tempfile.NamedTemporaryFile(delete=False, suffix='.pdf') as temp_file:
        temp_file.write(pdf_bytes)
        temp_file_path = temp_file.name

    try:
        # 2. 读取 PDF 文本
        loader = PyPDFLoader(temp_file_path)
        documents = loader.load()

        # 3. 切块
        splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
        docs = splitter.split_documents(documents)

        # 4. 嵌入模型（可替换为 OpenAI 或其他）
        embeddings = HuggingFaceEmbeddings(model_name="/home/yundai/AI_notes/agents_rags/model")

        # 5. 创建 FAISS 向量库
        vectorstore = FAISS.from_documents(docs, embeddings)

        # 6. 保存到磁盘
        vectorstore.save_local(index_path)
    finally:
        os.unlink(temp_file_path)

    return index_path


In [4]:
with open("/home/yundai/AI_notes/agents_rags/合同样例.pdf", "rb") as f:
    pdf_bytes = f.read()
process_pdf_to_faiss(pdf_bytes, index_path="faiss_index")

/tmp/ipykernel_1129004/1508565259.py:30: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="/home/yundai/AI_notes/agents_rags/model")
/home/yundai/tfenv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


'faiss_index'

In [5]:
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

# 加载本地 FAISS
embeddings = HuggingFaceEmbeddings(model_name="/home/yundai/AI_notes/agents_rags/model")
vectorstore = FAISS.load_local("faiss_index", embeddings, allow_dangerous_deserialization=True)

# 搜索
query = "文档中提到的主要技术有哪些？"
docs = vectorstore.similarity_search(query, k=3)

# 输出
for i, doc in enumerate(docs, 1):
    print(f"--- 结果 {i} ---")
    print(doc.page_content)


--- 结果 1 ---
- 4 -
清款项并获得作品所有权后，应合理使用作品，不得侵犯乙方的署名
权。
七、违约责任
7.1任何一方未按本合同的约定履行自身义务的，任一方不得无
故终止合同。
7.2协议履行期间，如甲方违约，乙方为维权所发生的任何费用
（包括但不限于赔偿金、违约金、诉讼费、公证费、律师费等）均由
甲方承担。
八、保密条款
8.1双方同意，本合同之条款、该等条款之存在、以及双方为本
合同书之目的提供予对方之任何资料、数据（包括但不限于以各种形
式、格式及媒介存在之程序、文档、技术文件，及其他甲方标明「专
有」或「机密」或类似注记之数据），属于商业秘密，仅限揭露予必
要之经理人、董事、员工、会计师、法律顾问及其它顾问，除适用法
律强制要求或政府主管机关之主动要求外，不得揭露予其它第三方，
且揭露时仅限与本交易相关「必要知悉」之信息，并应保密。
8.2本合同约定保密义务不因合同的终止/解除而失效。
8.3除本合同另有约定外，甲乙双方均同意对对方的任何资料负
绝对之保密义务及保管责任，未经资料提供方事先书面同意，绝不作
超出本合同书之目的范围之使用或以任何方式将其泄露、告知、交付
予任何第三人，若有违反致一方或一方之客户受有损害，同意无条件
赔偿该方所受之一切损害（包括但不限于合理的律师费、维权费用、
诉讼费、公证费、差旅费等），如另涉有民刑事责任，并应负起相关
所有民刑事责任。
--- 结果 2 ---
- 2 -
1.4 乙方为甲方开发的小程序，不允许用于传销、诈骗、赌博、
色情等任何非法领域。如甲方私自修改乙方开发的小程序并用于上述
非法领域，一切责任由甲方承担。 .
二、制作项目及单价
2.12.1本合同总价款：合同总价款为人民币 2800 元。
合同签订后支付 50%，即 1400 元，开发时间共五天，第六天交
付支付尾款1400 元。
2.2合作期限： 2024 年 12 月 17 至 2024 年 12
月 24 日止。
2.3付款方式：通过第 2.3.2 种方式支付款项。
2.3.1甲方通过淘宝交易的方式向乙方支付款项。
2.3.2甲方通过微信/支付宝/转账方式向乙方支付款项，乙方应
在收到款项后向甲方开具等额发票。
2.3.3若甲方逾期未支付款项，乙方有权暂停服务，并要求甲方
按应付未付款金额的日万分之五支付违约金。
三、验收标准和方法


In [6]:
knowledge_base = vectorstore

In [7]:
legal_researcher = Agent(
            name="Legal Researcher",
            role="Legal research specialist",
            model=llm,
            tools=[DuckDuckGoTools()],
            knowledge=knowledge_base,
            search_knowledge=True,
            instructions=[
                "Find and cite relevant legal cases and precedents",
                "Provide detailed research summaries with sources",
                "Reference specific sections from the uploaded document",
                "Always search the knowledge base for relevant information"
            ],
            show_tool_calls=True,
            markdown=True,
            memory=None,
        )

contract_analyst = Agent(
    name="Contract Analyst",
    role="Contract analysis specialist",
    model=llm,
    knowledge=knowledge_base,
    search_knowledge=True,
    instructions=[
        "Review contracts thoroughly",
        "Identify key terms and potential issues",
        "Reference specific clauses from the document"
    ],
    markdown=True,
    memory=None,
)

legal_strategist = Agent(
    name="Legal Strategist", 
    role="Legal strategy specialist",
    model=llm,
    knowledge=knowledge_base,
    search_knowledge=True,
    instructions=[
        "Develop comprehensive legal strategies",
        "Provide actionable recommendations",
        "Consider both risks and opportunities"
    ],
    markdown=True,
    memory=None,
)

# Legal Agent Team
legal_team = Agent(
    name="Legal Team Lead",
    role="Legal team coordinator",
    model=llm,
    team=[legal_researcher, contract_analyst, legal_strategist],
    knowledge=knowledge_base,
    search_knowledge=True,
    instructions=[
        "Coordinate analysis between team members",
        "Provide comprehensive responses",
        "Ensure all recommendations are properly sourced",
        "Reference specific parts of the uploaded document",
        "Always search the knowledge base before delegating tasks"
    ],
    show_tool_calls=True,
    markdown=True,
    memory=None,
)

In [8]:
analysis_configs = {
            "Contract Review": {
                "query": "Review this contract and identify key terms, obligations, and potential issues.",
                "agents": ["Contract Analyst"],
                "description": "Detailed contract analysis focusing on terms and obligations"
            },
            "Legal Research": {
                "query": "Research relevant cases and precedents related to this document.",
                "agents": ["Legal Researcher"],
                "description": "Research on relevant legal cases and precedents"
            },
            "Risk Assessment": {
                "query": "Analyze potential legal risks and liabilities in this document.",
                "agents": ["Contract Analyst", "Legal Strategist"],
                "description": "Combined risk analysis and strategic assessment"
            },
            "Compliance Check": {
                "query": "Check this document for regulatory compliance issues.",
                "agents": ["Legal Researcher", "Contract Analyst", "Legal Strategist"],
                "description": "Comprehensive compliance analysis"
            },
            "Custom Query": {
                "query": None,
                "agents": ["Legal Researcher", "Contract Analyst", "Legal Strategist"],
                "description": "Custom analysis using all available agents"
            }
        }

In [9]:
user_query = "帮我进行法律分析"

In [10]:
analysis_type = "Compliance Check"

In [ ]:
from typing import List, Dict, Any
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, END


class LegalState(TypedDict, total=False):
    query: str
    docs: List[str]
    results: Dict[str, str]
    selected_agents: List[str]
    final_answer: str


def build_legal_graph(llm, vectorstore):
    """
    使用 LangGraph 构建法律分析工作流：检索 -> (研究/合同/策略 顺序执行, 仅对选择的角色生效) -> 合并
    """
    RETRIEVE_K = 4

    def retrieve_node(state: LegalState) -> Dict[str, Any]:
        query = state.get("query", "")
        docs = vectorstore.similarity_search(query, k=RETRIEVE_K)
        texts = [d.page_content for d in docs]
        return {"docs": texts}

    def _run_role(state: LegalState, role_name: str, role_instructions: List[str]) -> Dict[str, Any]:
        if role_name not in state.get("selected_agents", []):
            return {}
        docs_text = "\n\n".join([f"- {t}" for t in state.get("docs", [])])
        instructions_text = "\n".join([f"- {it}" for it in role_instructions])
        prompt = f"""You are {role_name}.
                Instructions:
                {instructions_text}

                Use the following document excerpts as context:
                {docs_text}

                Task:
                {state.get('query', '')}

                Provide a concise, well-structured analysis with citations to the excerpts when relevant."""
        try:
            msg = llm.invoke(prompt)
            content = getattr(msg, "content", None) or str(msg)
        except Exception as e:
            content = f"{role_name} failed: {e}"
        merged = dict(state.get("results", {}))
        merged[role_name] = content
        return {"results": merged}

    def legal_researcher_node(state: LegalState) -> Dict[str, Any]:
        return _run_role(
            state,
            "Legal Researcher",
            [
                "Find and cite relevant legal cases and precedents",
                "Provide detailed research summaries with sources",
                "Reference specific sections from the uploaded document",
            ],
        )

    def contract_analyst_node(state: LegalState) -> Dict[str, Any]:
        return _run_role(
            state,
            "Contract Analyst",
            [
                "Review contracts thoroughly",
                "Identify key terms and potential issues",
                "Reference specific clauses from the document",
            ],
        )

    def legal_strategist_node(state: LegalState) -> Dict[str, Any]:
        return _run_role(
            state,
            "Legal Strategist",
            [
                "Develop comprehensive legal strategies",
                "Provide actionable recommendations",
                "Consider both risks and opportunities",
            ],
        )

    def combine_node(state: LegalState) -> Dict[str, Any]:
        docs_text = "\n\n".join([f"- {t}" for t in state.get("docs", [])])
        role_outputs = state.get("results", {})
        parts = [f"### {k}\n{v}" for k, v in role_outputs.items()]
        combined = "\n\n".join(parts) if parts else "No role outputs."
        synth_prompt = f"""Synthesize a final legal analysis from the role outputs below.

                    Context excerpts:
                    {docs_text}

                    Role outputs:
                    {combined}

                    Deliver:
                    - Detailed analysis
                    - Key points (bullets)
                    - Recommendations (bullets)
                    """
        try:
            msg = llm.invoke(synth_prompt)
            final_answer = getattr(msg, "content", None) or str(msg)
        except Exception:
            final_answer = combined
        return {"final_answer": final_answer}

    graph = StateGraph(LegalState)
    graph.add_node("retrieve", retrieve_node)
    graph.add_node("legal_researcher", legal_researcher_node)
    graph.add_node("contract_analyst", contract_analyst_node)
    graph.add_node("legal_strategist", legal_strategist_node)
    graph.add_node("combine", combine_node)

    graph.set_entry_point("retrieve")
    graph.add_edge("retrieve", "legal_researcher")
    graph.add_edge("legal_researcher", "contract_analyst")
    graph.add_edge("contract_analyst", "legal_strategist")
    graph.add_edge("legal_strategist", "combine")
    graph.add_edge("combine", END)

    return graph.compile()


In [12]:
combined_query = f"""
                Using the uploaded document as reference:
                
                {user_query}
                
                Please search the knowledge base and provide specific references from the document.
                Focus Areas: {', '.join(analysis_configs[analysis_type]['agents'])}
                """

In [13]:
# 构建并运行 LangGraph 工作流
legal_graph = build_legal_graph(llm=llm, vectorstore=vectorstore)

# 将原 combined_query 和 agent 选择映射到图的状态
selected_agents = analysis_configs[analysis_type]["agents"]  # 如 ["Legal Researcher", ...]

state_input = {
    "query": combined_query,
    "selected_agents": selected_agents,
}

# 运行图，获取最终结果
final_state = legal_graph.invoke(state_input)

# 与原接口保持兼容：构造一个简易 response 对象
class _Response:
    def __init__(self, content: str):
        self.content = content
        self.messages = []

response = _Response(content=final_state.get("final_answer", ""))


In [14]:
response.__dict__

{'content': '### **Final Legal Analysis & Synthesis**  \n\n#### **1. Core Legal Issues**  \n**(a) Intellectual Property (IP) Rights**  \n- **Ownership Transfer**: IP (copyright, publication rights) transfers to **甲方 (Party A)** only upon **full payment** (Clause 6). Until then, **乙方 (Party B)** retains all rights, including the right to demand cessation of use (Clause 5.5).  \n- **Attribution Rights**: Party B retains **署名权 (right of attribution)** perpetually, which Party A must respect (Clauses 4, 6).  \n- **Risk**: Unauthorized use by Party A before payment constitutes IP infringement, triggering liability for damages (Clause 7.2).  \n\n**(b) Payment & Modifications**  \n- Party A bears **additional costs** for mid-project changes (e.g., design/spec revisions) (Clause 4.4).  \n- Party B may reject **out-of-scope modifications** or charge extra fees (Clause 5.3).  \n- **Risk**: Ambiguity in "reasonable" modifications could lead to disputes.  \n\n**(c) Confidentiality**  \n- Strict **

In [15]:
response.content

'### **Final Legal Analysis & Synthesis**  \n\n#### **1. Core Legal Issues**  \n**(a) Intellectual Property (IP) Rights**  \n- **Ownership Transfer**: IP (copyright, publication rights) transfers to **甲方 (Party A)** only upon **full payment** (Clause 6). Until then, **乙方 (Party B)** retains all rights, including the right to demand cessation of use (Clause 5.5).  \n- **Attribution Rights**: Party B retains **署名权 (right of attribution)** perpetually, which Party A must respect (Clauses 4, 6).  \n- **Risk**: Unauthorized use by Party A before payment constitutes IP infringement, triggering liability for damages (Clause 7.2).  \n\n**(b) Payment & Modifications**  \n- Party A bears **additional costs** for mid-project changes (e.g., design/spec revisions) (Clause 4.4).  \n- Party B may reject **out-of-scope modifications** or charge extra fees (Clause 5.3).  \n- **Risk**: Ambiguity in "reasonable" modifications could lead to disputes.  \n\n**(c) Confidentiality**  \n- Strict **NDA terms** 

In [16]:
from IPython.display import display, Markdown

def md(text: str):
    display(Markdown(text))

# 高可读性中间过程展示
md("### 任务参数")
md(f"- **分析类型**: `{analysis_type}`")
md(f"- **参与角色**: {', '.join(selected_agents)}")

# 展示检索到的上下文
md("### 检索到的上下文片段")
docs = final_state.get("docs", [])
if not docs:
    md("> 未检索到内容")
else:
    for i, t in enumerate(docs, 1):
        preview = t.strip()
        if len(preview) > 800:
            preview = preview[:800] + "…"
        md(f"**片段 {i}**\n\n> {preview}")

# 展示各角色的阶段性输出
md("### 角色输出（阶段性结果）")
role_outputs = final_state.get("results", {})
if not role_outputs:
    md("> 无角色输出")
else:
    for role_name, content in role_outputs.items():
        md(f"#### {role_name}")
        md(content if content else "> 无内容")

# 展示最终合成结论
md("### 最终合成结论")
md(final_state.get("final_answer", "> 无最终结论"))


### 任务参数

- **分析类型**: `Compliance Check`

- **参与角色**: Legal Researcher, Contract Analyst, Legal Strategist

### 检索到的上下文片段

**片段 1**

> - 4 -
清款项并获得作品所有权后，应合理使用作品，不得侵犯乙方的署名
权。
七、违约责任
7.1任何一方未按本合同的约定履行自身义务的，任一方不得无
故终止合同。
7.2协议履行期间，如甲方违约，乙方为维权所发生的任何费用
（包括但不限于赔偿金、违约金、诉讼费、公证费、律师费等）均由
甲方承担。
八、保密条款
8.1双方同意，本合同之条款、该等条款之存在、以及双方为本
合同书之目的提供予对方之任何资料、数据（包括但不限于以各种形
式、格式及媒介存在之程序、文档、技术文件，及其他甲方标明「专
有」或「机密」或类似注记之数据），属于商业秘密，仅限揭露予必
要之经理人、董事、员工、会计师、法律顾问及其它顾问，除适用法
律强制要求或政府主管机关之主动要求外，不得揭露予其它第三方，
且揭露时仅限与本交易相关「必要知悉」之信息，并应保密。
8.2本合同约定保密义务不因合同的终止/解除而失效。
8.3除本合同另有约定外，甲乙双方均同意对对方的任何资料负
绝对之保密义务及保管责任，未经资料提供方事先书面同意，绝不作
超出本合同书之目的范围之使用或以任何方式将其泄露、告知、交付
予任何第三人，若有违反致一方或一方之客户受有损害，同意无条件
赔偿该方所受之一切损害（包括但不限于合理的律师费、维权费用、
诉讼费、公证费、差旅费等），如另涉有民刑事责任，并应负起相关
所有民刑事责任。

...

### **Legal Analysis of the Contract**  

#### **1. Key Terms & Obligations**  
- **Ownership & Intellectual Property (IP)**:  
  - IP rights transfer to **甲方 (Party A)** only upon full payment (Clause 6). Before payment, **乙方 (Party B)** retains all rights (e.g., copyright, publication rights) (Clause 6).  
  - Party A must respect Party B’s **attribution rights** even after ownership transfer (Clause 4).  

- **Payment & Deliverables**:  
  - Party A must pay in full to gain usage/modification rights (Clause 4.3). Unauthorized use before payment allows Party B to demand cessation and fees (Clause 5.5).  
  - Party A bears costs for **mid-project changes** (e.g., design/spec revisions) (Clause 4.4).  

- **Performance & Deadlines**:  
  - Party B must deliver work **on time** and adhere to agreed specifications (Clause 5.1–5.2).  
  - Party B may reject **out-of-scope modifications** or charge extra (Clause 5.3).  

#### **2. Potential Legal Risks & Issues**  
- **Breach of Contract**:  
  - **Termination Restrictions**: Neither party may terminate without cause (Clause 7.1), but Party A bears all legal/liability costs if it breaches (Clause 7.2).  
  - **Subcontracting**: Party B may delegate obligations to third parties (Clause 5.4), but the contract lacks clarity on quality control or liability for subcontractor failures.  

- **Confidentiality Risks**:  
  - Strict **NDA terms** apply indefinitely (Clause 8.2), with liability for leaks (including legal fees) (Clause 8.3).  
  - Ambiguity exists on what constitutes "necessary disclosure" to advisors (Clause 8.1).  

- **Force Majeure**:  
  - Covers natural disasters/policy changes but requires **proof within 10 days** (Clause 9.1). Delays in notification could void protections.  

#### **3. Dispute Resolution & Jurisdiction**  
- **Governing Law**: Based on China’s Civil Code (Preamble).  
- **Dispute Venue**: Exclusive jurisdiction at **Party B’s local court** (Clause 10.2), which may disadvantage Party A.  

#### **4. Recommendations**  
- **For Party A**:  
  - Ensure **explicit approval rights** over subcontractors (Clause 5.4).  
  - Clarify "reasonable use" of IP post-payment to avoid attribution disputes (Clause 6).  
- **For Party B**:  
  - Define **"necessary disclosure"** scope in Clause 8.1 to prevent overreach.  
  - Specify **force majeure documentation standards** (Clause 9.1).  

**Citations**: Referenced clauses are marked (e.g., Clause 4.3) from the provided excerpts.  

---  
**Conclusion**: The contract is heavily protective of Party B’s IP and confidentiality, with stringent penalties for Party A’s breaches. Key risks include ambiguous subcontracting terms and one-sided dispute resolution. Parties should negotiate clearer terms on modifications, third-party involvement, and confidentiality exceptions.

#### Legal Strategist

### **Legal Analysis of the Contract Between 深圳市亘古磐石科技有限责任公司 (甲方) and 北京天虹广告设计有限公司 (乙方)**  

#### **1. Key Legal Risks & Obligations**  
**(a) Intellectual Property (IP) Ownership & Usage (Excerpts: -3-, -4-)**  
- **甲方's Rights**: Full ownership (copyright, publication rights, etc.) transfers only upon full payment (4.3, 6). Prior to payment, 乙方 retains all IP rights (5.5).  
- **乙方's Rights**: Retains **attribution rights** (署名权) even after transfer (6). Unauthorized use by 甲方 before payment allows 乙方 to demand cessation and payment (5.5).  
- **Risk for 甲方**: Using unfinished/unpaid work risks breach (5.5) and liability for damages (7.2).  

**(b) Payment & Modification Terms (Excerpts: -3-)**  
- 甲方 must pay additional costs for mid-project changes (4.4).  
- 乙方 may reject modifications beyond contract scope or charge extra fees (5.3).  
- **Risk**: Unclear modification requests could lead to disputes over "reasonable" adjustments.  

**(c) Confidentiality (Excerpts: -4-)**  
- Strict nondisclosure applies to contract terms, shared data, and proprietary materials (8.1).  
- Breach triggers liability for damages (e.g., legal fees) and potential civil/criminal penalties (8.3).  
- **Risk**: Unauthorized subcontracting (5.4) or third-party disclosures could violate confidentiality.  

#### **2. Enforcement & Dispute Resolution**  
**(a) Termination & Breach (Excerpts: -4-)**  
- Neither party may terminate without cause (7.1).  
- 甲方 bears all costs (litigation, attorneys’ fees) if违约 (7.2).  

**(b) Jurisdiction (Excerpts: -5-)**  
- Disputes resolved via negotiation; failing this, 乙方’s local court has jurisdiction (10.2).  
- **Strategic Note**: Favors 乙方 geographically; 甲方 may face higher litigation costs.  

#### **3. Force Majeure & Flexibility (Excerpts: -5-)**  
- Covers natural disasters, policy changes, etc. (9.1).  
- Parties must notify within **5 days** and provide proof within **10 days** to claim exemption.  

#### **4. Actionable Recommendations**  
1. **For 甲方**:  
   - Ensure **full payment** before using deliverables to avoid IP infringement (5.5).  
   - Document all modification requests to justify additional fees (4.4, 5.3).  

2. **For 乙方**:  
   - Enforce attribution rights post-payment (6) and monitor unauthorized use (5.5).  
   - Limit subcontracting to avoid confidentiality breaches (8.1 + 5.4).  

3. **For Both**:  
   - Define "reasonable" modifications in a **supplemental agreement** (10.1) to reduce ambiguity.  
   - Maintain records of force majeure events (9.1) to mitigate liability.  

**Citation Summary**: Risks center on **IP transfer timing**, **confidentiality**, and **modification costs**, with strict penalties for breach (7.2, 8.3). Proactive documentation and adherence to notice periods (9.1) are critical.  

---  
*Analysis based on excerpts from the contract, focusing on enforceable terms under Chinese contract law (《民法典》).*

### 最终合成结论

### **Final Legal Analysis & Synthesis**  

#### **1. Core Legal Issues**  
**(a) Intellectual Property (IP) Rights**  
- **Ownership Transfer**: IP (copyright, publication rights) transfers to **甲方 (Party A)** only upon **full payment** (Clause 6). Until then, **乙方 (Party B)** retains all rights, including the right to demand cessation of use (Clause 5.5).  
- **Attribution Rights**: Party B retains **署名权 (right of attribution)** perpetually, which Party A must respect (Clauses 4, 6).  
- **Risk**: Unauthorized use by Party A before payment constitutes IP infringement, triggering liability for damages (Clause 7.2).  

**(b) Payment & Modifications**  
- Party A bears **additional costs** for mid-project changes (e.g., design/spec revisions) (Clause 4.4).  
- Party B may reject **out-of-scope modifications** or charge extra fees (Clause 5.3).  
- **Risk**: Ambiguity in "reasonable" modifications could lead to disputes.  

**(c) Confidentiality**  
- Strict **NDA terms** cover contract terms, proprietary data, and materials marked as confidential (Clause 8.1).  
- Obligations survive contract termination (Clause 8.2).  
- **Penalties**: Breach triggers liability for **all damages** (including legal fees) and potential civil/criminal penalties (Clause 8.3).  

**(d) Breach & Dispute Resolution**  
- **Termination Restrictions**: Neither party may terminate without cause (Clause 7.1).  
- **Remedies**: Party A bears all costs (litigation, attorneys’ fees) if it breaches (Clause 7.2).  
- **Jurisdiction**: Disputes resolved in **Party B’s local court** (Clause 10.2), favoring Party B geographically.  

**(e) Force Majeure**  
- Covers natural disasters, policy changes, etc. (Clause 9.1).  
- **Procedure**: Affected party must notify within **5 days** and provide proof within **10 days** to claim exemption (Clause 9.1).  

---

#### **2. Key Recommendations**  
**For Party A (甲方):**  
- **Ensure full payment** before using deliverables to avoid IP infringement (Clause 5.5).  
- **Document all modification requests** to justify additional fees (Clauses 4.4, 5.3).  
- **Negotiate subcontractor approval rights** to mitigate confidentiality risks (Clause 5.4).  

**For Party B (乙方):**  
- **Enforce attribution rights** post-payment (Clause 6) and monitor unauthorized use (Clause 5.5).  
- **Limit subcontracting** to avoid confidentiality breaches (Clauses 5.4, 8.1).  
- **Clarify "necessary disclosure"** in confidentiality clauses to prevent overreach (Clause 8.1).  

**For Both Parties:**  
- **Define "reasonable modifications"** in a **supplemental agreement** (Clause 10.1) to reduce ambiguity.  
- **Maintain records** of force majeure events to comply with notice requirements (Clause 9.1).  
- **Prioritize negotiation** before litigation to avoid unfavorable jurisdiction (Clause 10.2).  

---

#### **3. Strategic Summary**  
- **IP & Payment**: Party A assumes significant risk if it uses deliverables before payment. Party B retains strong leverage through IP retention and attribution rights.  
- **Confidentiality**: Strict terms protect both parties but require careful third-party management.  
- **Dispute Resolution**: Party B’s jurisdictional advantage incentivizes Party A to resolve conflicts early.  
- **Force Majeure**: Timely documentation is critical to invoke protections.  

**Citations**: All references are to the provided contract excerpts (Sections 4–10). For external context, cross-reference with China’s **《民法典》 (Civil Code)** on contract and IP law.  

**Final Note**: This contract heavily favors Party B’s protections (IP, confidentiality, jurisdiction). Party A should negotiate clearer terms on modifications, subcontracting, and dispute venue where possible.